[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch07-frameworks/04-tensor-va-du-lieu.ipynb)

# Tensor, thiết bị và dữ liệu: những hợp đồng dưới một lời gọi `model(x)`

Khi bạn gọi `model(x)`, framework phải quyết định năm việc trước khi kernel đầu tiên chạy: biểu diễn dữ
liệu ra sao, đặt nó ở thiết bị nào, đưa nó vào kịp không, tổ chức tham số thế nào, và chạy kernel nào.
Mỗi quyết định là một hợp đồng ngầm về hình dạng, kiểu dữ liệu, bố trí, thiết bị và đơn vị. Sổ tay này
mở từng hợp đồng ra, tính những con số của chương, và đo những gì đo được trên CPU của bạn.

**Bạn sẽ làm:**
1. tính hệ số 4.45 của tàu Mars Climate Orbiter, rồi tạo lại lỗi cùng loại với một tensor ảnh;
2. đọc siêu dữ liệu, sải bước, view và bản sao của tensor bằng tay, và đo cái giá của một phép chuyển vị;
3. dùng kiểu dữ liệu như một đòn bẩy: byte, sai số làm tròn, INT8 bằng tay, và tràn dưới của FP16;
4. tính thời gian truyền qua các đường truyền của chương, đo đường copy trên máy này, và tính nhịp mà quy
   trình dữ liệu phải giữ;
5. kiểm các nguyên lý của `nn.Module`, rồi đặt một mô hình nhỏ lên đích triển khai 256 KB RAM.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import glob
import io
from collections import Counter

import pandas as pd
import torch.fx as fx
import torch.nn.functional as F
import torchvision
from torch import nn
from torch.fx.passes.shape_prop import ShapeProp
from torchvision import transforms

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # bốn màu cố định cho các chuỗi số liệu


def thoi_gian(viec, lan=5):
    """Thời gian nhỏ nhất của `lan` lần chạy (giây): ít bị nhiễu bởi việc khác trên máy nhất."""
    viec()
    tot = float("inf")
    for _ in range(lan):
        t0 = time.perf_counter()
        viec()
        tot = min(tot, time.perf_counter() - t0)
    return tot


print("sẵn sàng")

## 1. Một hợp đồng ngầm: đơn vị

Năm 1999, phần mềm mặt đất của tàu Mars Climate Orbiter tính xung lực đẩy theo pound-lực nhân giây,
còn phần mềm dẫn đường đọc con số đó như newton nhân giây. Hệ số giữa hai đơn vị suy ra được từ hai định
nghĩa quốc tế: một pound bằng đúng 0.45359237 kg, và gia tốc trọng trường chuẩn bằng đúng 9.80665 m/s².

In [ ]:
POUND_KG = 0.45359237      # định nghĩa quốc tế của pound
G_CHUAN = 9.80665          # gia tốc trọng trường chuẩn (m/s²), cũng là một định nghĩa
he_so = POUND_KG * G_CHUAN  # 1 lbf·s tính bằng N·s
theo_sach("1 pound-lực giây bằng bao nhiêu newton giây", he_so, sach=4.45, nguon="4,45×")
can_diem, ke_hoach = sach(57, nguon="57 km so với 226 km"), sach(226, nguon="57 km so với 226 km")
print(f"cận điểm thực tế {can_diem} km so với kế hoạch {ke_hoach} km: thấp hơn {ke_hoach - can_diem} km")

Cả hai phần mềm đều đúng, chỉ có hợp đồng giữa chúng là ngầm. Chương nói ML gặp đúng loại lỗi này khi
bố trí bộ nhớ (memory layout) của tensor, như NCHW với NHWC, khác nhau giữa hai công đoạn. Một batch ảnh
lưu theo NHWC (kênh nằm cuối) được đưa vào một tầng tích chập chờ NCHW. Có hai cách "sửa hình dạng":
`permute` sắp lại các trục, còn `reshape` chỉ cắt lại cùng một dãy số theo hình dạng mới.

**Dự đoán:** cách nào báo lỗi?

In [ ]:
dat_hat_giong()
anh_nhwc = torch.rand(2, 8, 8, 3)                       # batch, cao, rộng, kênh
dung = anh_nhwc.permute(0, 3, 1, 2)                     # đổi trục thật: N, C, H, W
sai = anh_nhwc.reshape(2, 3, 8, 8)                      # cùng hình dạng, sai nội dung
assert dung.shape == sai.shape
tich_chap = nn.Conv2d(3, 4, 3)
ra_dung, ra_sai = tich_chap(dung), tich_chap(sai)       # cả hai đều chạy, không lỗi nào
print("hình dạng giống nhau:", tuple(dung.shape), "· số phần tử khác nhau:", int((dung != sai).sum()), "/", dung.numel())
print(f"đầu ra tích chập lệch nhau trung bình {(ra_dung - ra_sai).abs().mean().item():.3f} "
      f"trên độ lớn trung bình {ra_dung.abs().mean().item():.3f}")
assert not torch.allclose(ra_dung, ra_sai)

Không cách nào báo lỗi. Hình dạng là thứ duy nhất framework kiểm được, và hình dạng khớp. Đầu ra trông
hợp lý và sai.

## 2. Siêu dữ liệu, sải bước, view và bản sao

Mỗi tensor mang siêu dữ liệu tensor (tensor metadata): hình dạng, kiểu dữ liệu, thiết bị, và sải bước
(stride), tức số phần tử phải bỏ qua trong bộ nhớ tuyến tính để đi một bước theo mỗi trục. Từ đó tính
được chính xác số byte.

In [ ]:
batch_anh = torch.zeros(sach(64, nguon="64, 3, 224, 224"), sach(3, nguon="64, 3, 224, 224"),
                        sach(224, nguon="64, 3, 224, 224"), sach(224, nguon="64, 3, 224, 224"))
print("hình dạng", tuple(batch_anh.shape), "· kiểu", batch_anh.dtype, "· thiết bị", batch_anh.device,
      "· sải bước", batch_anh.stride())
theo_sach("batch ảnh FP32 (MB)", batch_anh.numel() * batch_anh.element_size() / 1e6, sach=38.5,
          nguon="batch 64 FP32 224×224×3 = 38,5 MB → 1,2 ms")
ma_tran = torch.zeros(sach(1024, trich="\\([1024, 1024]\\)"), sach(1024, trich="\\([1024, 1024]\\)"))
theo_sach("ma trận 1024 × 1024 FP32 (byte)", ma_tran.numel() * ma_tran.element_size(), sach=4194304,
          trich="4{,}194{,}304")
theo_sach("cùng ma trận đó (MB)", ma_tran.numel() * ma_tran.element_size() / 1e6, sach=4.2, trich="(about 4.2 MB)")
for hang_ in range(6):                                  # hạng 0 tới 5: từ số vô hướng tới batch video
    t_ = torch.zeros((2,) * hang_)
    print(f"hạng {hang_}: hình dạng {tuple(t_.shape)}, sải bước {t_.stride()}")

### Sải bước bằng tay

Chương lấy ví dụ một tensor 2×3. Ưu tiên hàng (row-major) lưu từng hàng liền nhau, nên sải bước là
$(3, 1)$. Ưu tiên cột (column-major) lưu từng cột liền nhau, nên sải bước đảo lại. Địa chỉ của phần tử
$(i, j)$ trong vùng nhớ là $i \cdot s_0 + j \cdot s_1$, và ô dưới kiểm công thức đó với chính vùng nhớ của
PyTorch.

In [ ]:
hang_t, cot_t = sach(2, nguon="2×3 · sải bước 3 và 1"), sach(3, nguon="2×3 · sải bước 3 và 1")
A = torch.arange(hang_t * cot_t, dtype=torch.float32).reshape(hang_t, cot_t)
A_cot = A.t().contiguous().t()                          # cùng giá trị, lưu theo cột
print("A =", A.tolist())
print("ưu tiên hàng: sải bước", A.stride(), "· vùng nhớ", A.flatten().tolist())
print("ưu tiên cột:  sải bước", A_cot.stride(), "· vùng nhớ", A_cot.t().flatten().tolist())
assert A.stride() == (sach(3, nguon="2×3 · sải bước 3 và 1"), sach(1, nguon="2×3 · sải bước 3 và 1"))
assert A_cot.stride() == (1, hang_t)
assert torch.equal(A, A_cot)                            # cùng tensor về toán học


def dia_chi(t, *chi_so):
    """Vị trí của một phần tử trong vùng nhớ: tổng chỉ số nhân sải bước, cộng độ lệch của view."""
    return t.storage_offset() + sum(i * s for i, s in zip(chi_so, t.stride()))


vung_nho = A_cot.untyped_storage()
gia_tri = torch.tensor([], dtype=torch.float32).set_(vung_nho)
for i in range(hang_t):
    for j in range(cot_t):
        assert gia_tri[dia_chi(A_cot, i, j)] == A_cot[i, j]
print("công thức địa chỉ khớp với vùng nhớ ở cả", hang_t * cot_t, "phần tử")

### View hay bản sao

Chương cảnh báo rằng một phép trên tensor có thể trả về một vùng nhớ mới, một view dùng chung vùng nhớ
cũ, hoặc sửa tại chỗ, và việc tính bộ nhớ phải phân biệt ba trường hợp. Ô dưới hỏi thẳng vùng nhớ: hai
tensor dùng chung một vùng khi `data_ptr` của vùng nhớ bằng nhau.

In [ ]:
X = torch.arange(12.0).reshape(3, 4)
cung_vung = lambda a, b: a.untyped_storage().data_ptr() == b.untyped_storage().data_ptr()
hang = []
for ten, y_ in (("X.t()", X.t()), ("X[:, 1:3]", X[:, 1:3]), ("X.view(4, 3)", X.view(4, 3)),
                ("X.t().contiguous()", X.t().contiguous()), ("X.t().reshape(12)", X.t().reshape(12)),
                ("X * 1", X * 1)):
    hang.append((ten, tuple(y_.shape), y_.stride(), y_.is_contiguous(), "view" if cung_vung(X, y_) else "bản sao"))
print(pd.DataFrame(hang, columns=["phép", "hình dạng", "sải bước", "liền mạch", "vùng nhớ"]).to_string(index=False))
try:
    X.t().view(12)
except RuntimeError as loi:
    print("X.t().view(12) bị từ chối:", str(loi).split(".")[0][:100])
v_ = X[:, 1:3]
v_ += 100                                                # sửa tại chỗ qua một view
print("sau khi cộng 100 vào view X[:, 1:3], hàng đầu của X là", X[0].tolist())
assert X[0, 1] == 101

BYTE_CPU = sach(4, nguon="4 byte → 128 byte")
BYTE_TANG_TOC = sach(128, nguon="4 byte → 128 byte")
moi = torch.zeros(1024)
lech = moi[1:]                                          # view lệch 4 byte so với đầu vùng nhớ
for ten, t_ in (("tensor mới cấp", moi), ("view moi[1:]", lech)):
    print(f"{ten}: địa chỉ chia hết cho {BYTE_CPU}: {t_.data_ptr() % BYTE_CPU == 0}, "
          f"cho {BYTE_TANG_TOC}: {t_.data_ptr() % BYTE_TANG_TOC == 0}")

Chuyển vị và cắt lát là view: chúng chỉ đổi sải bước và độ lệch, không chép byte nào, và một phép sửa tại
chỗ trên view đi thẳng vào tensor gốc. `view` từ chối một tensor không liền mạch, còn `reshape` âm thầm
chép nếu cần. Một view cắt lát còn có thể bắt đầu ở địa chỉ không căn lề theo yêu cầu của bộ tăng tốc,
mà chương nói có thể tới 128 byte.

### Cái giá của một bố trí lệch

Một view chuyển vị là miễn phí, cho tới khi có phép nào phải đọc nó. Ô dưới đo hai phép chép cùng số byte
vào cùng một vùng nhớ đã cấp sẵn: chép một ma trận liền mạch, và chép bản chuyển vị của nó, tức đọc theo
cột một ma trận lưu theo hàng.

**Dự đoán:** hai phép chép cùng số byte. Chúng có tốn cùng thời gian không?

In [ ]:
ket = []
for n_ in (128, 512, 1024, 2048):
    x_ = torch.randn(n_, n_)
    dich = torch.empty(n_, n_)
    t_lien = thoi_gian(lambda: dich.copy_(x_))
    t_chuyen_vi = thoi_gian(lambda: dich.copy_(x_.t()))
    ket.append((n_, 2 * x_.nbytes / t_lien / 1e9, 2 * x_.nbytes / t_chuyen_vi / 1e9))
for n_, bw_lien, bw_cv in ket:
    do_tren_may(f"{n_:>4} × {n_:<4} chép liền mạch", bw_lien, "GB/s")
    do_tren_may(f"{n_:>4} × {n_:<4} chép bản chuyển vị", bw_cv, "GB/s")

fig, ax = plt.subplots(figsize=(7, 3.6))
ns = [k[0] for k in ket]
ax.semilogx(ns, [k[1] for k in ket], "o-", color=MAU[0], ms=8, base=2, label="chép liền mạch")
ax.semilogx(ns, [k[2] for k in ket], "s--", color=MAU[1], ms=8, base=2, label="chép bản chuyển vị")
ax.set_xticks(ns, [f"{n_}²" for n_ in ns])
ax.set_ylim(bottom=0)
ax.set_xlabel("kích thước ma trận FP32")
ax.set_ylabel("băng thông hiệu dụng (GB/s, đọc + ghi)")
ax.set_title("Cùng số byte, hai bố trí (đo trên máy này)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Nếu đường chuyển vị nằm dưới, đó là vì mỗi lần đọc theo cột nhảy cả một hàng trong bộ nhớ, nên mỗi dòng
bộ nhớ đệm được nạp về chỉ dùng một phần. Chương nói không thứ tự nào tốt hơn trong mọi trường hợp: đọc
theo hàng một tensor lưu theo cột cũng chịu đúng cái giá ấy. Điều quyết định là phép toán duyệt theo
chiều nào, và kernel được viết cho bố trí nào.

## 3. Kiểu dữ liệu là một đòn bẩy

Kiểu dữ liệu cân dải số và độ chính xác với lượng byte phải di chuyển. Ô dưới đổi cùng một tensor sang
vài kiểu, đếm byte, đo sai số làm tròn, và lượng tử hóa (quantization) sang INT8 bằng tay: chọn hệ số co
giãn $s = \max|x| / 127$, lưu $q = \mathrm{round}(x / s)$, rồi khôi phục $\hat x = q \cdot s$.

In [ ]:
dat_hat_giong()
x_goc = torch.randn(10_000)
FP32_B, INT8_B = sach(4, nguon="4 byte → 1 byte · 4× · 2–4×"), sach(1, nguon="4 byte → 1 byte · 4× · 2–4×")
he_so_cg = x_goc.abs().max() / 127                      # hệ số co giãn đối xứng, một cho cả tensor
q = torch.clamp(torch.round(x_goc / he_so_cg), -127, 127).to(torch.int8)
x_int8 = q.float() * he_so_cg
hang = []
for ten, x_ in (("FP32", x_goc), ("FP16", x_goc.half().float()), ("BF16", x_goc.bfloat16().float()), ("INT8", x_int8)):
    luu = {"FP32": x_goc, "FP16": x_goc.half(), "BF16": x_goc.bfloat16(), "INT8": q}[ten]
    hang.append((ten, luu.element_size(), (x_ - x_goc).abs().max().item(), ((x_ - x_goc).abs() / x_goc.abs()).median().item()))
print(pd.DataFrame(hang, columns=["kiểu", "byte mỗi số", "sai số lớn nhất", "sai số tương đối trung vị"])
      .to_string(index=False, float_format="{:.2e}".format))
theo_sach("FP32 sang INT8: giảm bộ nhớ (lần)", FP32_B / INT8_B, sach=4, nguon="4 byte → 1 byte · 4× · 2–4×")
assert (x_int8 - x_goc).abs().max() <= he_so_cg / 2 + 1e-6   # làm tròn: lệch tối đa nửa bước lượng tử
print(f"hệ số co giãn s = {he_so_cg.item():.4f}: sai số INT8 không vượt s / 2 = {he_so_cg.item() / 2:.4f}")

FP16 và BF16 cùng 2 byte nhưng chia bit khác nhau: FP16 dành nhiều bit cho phần định trị nên làm tròn
mịn hơn, BF16 giữ đủ bit mũ của FP32 nên giữ được dải số. Khác biệt về dải số là thứ quyết định khi huấn
luyện đa độ chính xác: một gradient rất nhỏ có thể tràn dưới (underflow) thành 0 ở FP16. Co giãn mất mát
(loss scaling) nhân mất mát lên trước lượt ngược để đẩy gradient vào dải biểu diễn được, rồi chia lại.
Ô dưới dùng hệ số $2^{16}$, giá trị khởi đầu mặc định của `GradScaler` trong PyTorch.

In [ ]:
gradient_nho = torch.tensor(1e-8)                        # một gradient ví dụ
HE_SO_CO_GIAN = 2.0**16
print("nhỏ nhất biểu diễn được: FP16", f"{torch.finfo(torch.float16).tiny:.1e} (chuẩn hoá),",
      f"BF16 {torch.finfo(torch.bfloat16).tiny:.1e}")
print("gradient 1e-8 ở FP16:", gradient_nho.half().item())
co_gian = (gradient_nho * HE_SO_CO_GIAN).half()
print(f"nhân {HE_SO_CO_GIAN:,.0f} rồi đổi sang FP16: {co_gian.item():.3e}; chia lại ở FP32: "
      f"{(co_gian.float() / HE_SO_CO_GIAN).item():.3e}")
print(f"gradient 1e-8 ở BF16, không co giãn: {gradient_nho.bfloat16().item():.3e}")
assert gradient_nho.half().item() == 0.0 and co_gian.item() > 0 and gradient_nho.bfloat16().item() > 0

Đó là lý do chương nói BF16 thường không cần co giãn mất mát: dải số của nó bằng FP32.

## 4. Đặt thiết bị, và phân cấp băng thông

Thời gian một lần truyền được mô hình hoá bằng $T = L_{lat} + D_{vol}/BW$: một độ trễ cố định mỗi lần,
cộng lượng dữ liệu chia băng thông. Bảng 7 của chương bỏ qua số hạng thứ nhất và tính số hạng thứ hai
cho một tensor 1000×1000 FP32, tức 4 MB, qua bốn đường truyền.

In [ ]:
D_4MB = sach(1000, trich="\\(1000{\\times}1000\\) float32 tensor (4 MB)") ** 2 * FP32_B
theo_sach("tensor 1000 × 1000 FP32 (MB)", D_4MB / 1e6, sach=4, trich="\\(1000{\\times}1000\\) float32 tensor (4 MB)")
duong = {   # đường truyền: (băng thông GB/s, thời gian của sách ms, nguồn)
    "PCIe 3.0 x16": (sach(15.8, nguon="15,8 GB/s · 0,254 ms"), sach(0.254, nguon="15,8 GB/s · 0,254 ms")),
    "PCIe 4.0 x16": (sach(32, nguon="32 GB/s · 0,125 ms"), sach(0.125, nguon="32 GB/s · 0,125 ms")),
    "NVLink 3.0": (sach(300, nguon="300 GB/s mỗi hướng (600 GB/s) · 0,013 ms"),
                   sach(0.013, nguon="300 GB/s mỗi hướng (600 GB/s) · 0,013 ms")),
    "bộ nhớ GPU": (sach(2039, nguon="2039 GB/s · 0,002 ms"), sach(0.002, nguon="2039 GB/s · 0,002 ms")),
}
for ten, (bw, t_sach) in duong.items():
    print(f"{ten:<13} {bw:>7,.1f} GB/s → {D_4MB / (bw * 1e9) * 1e3:.4f} ms   (sách ghi {t_sach} ms)")
theo_sach("PCIe 4.0 x16 (ms)", D_4MB / 32e9 * 1e3, sach=0.125, nguon="32 GB/s · 0,125 ms")
theo_sach("NVLink 3.0 (ms)", D_4MB / 300e9 * 1e3, sach=0.013, nguon="300 GB/s mỗi hướng (600 GB/s) · 0,013 ms")
theo_sach("bộ nhớ GPU (ms)", D_4MB / 2039e9 * 1e3, sach=0.002, nguon="2039 GB/s · 0,002 ms")
print(f"PCIe 3.0: 4 MB / 15.8 GB/s = {D_4MB / 15.8e9 * 1e3:.4f} ms, sách ghi 0.254 ms; "
      f"0.254 ms ứng với {D_4MB / 0.254e-3 / 1e9:.2f} GB/s")

Ba dòng khép đúng. Dòng PCIe 3.0 thì không: 4 MB chia 15.8 GB/s là 0.2532 ms, còn 0.254 ms ứng với khoảng
15.75 GB/s. Có lẽ thời gian được tính từ băng thông lý thuyết chưa làm tròn của PCIe 3.0 x16 (8 GT/s mỗi
làn, mã hoá 128b/130b, khoảng 15.75 GB/s), còn cột băng thông thì đã làm tròn lên 15.8. Hai số khác nhau ở
chữ số thứ ba, nên sổ tay in cả hai thay vì kiểm.

Chương còn nêu khoảng cách giữa PCIe 4.0 và bộ nhớ của GPU, một chặn dưới cho 1 GB qua PCIe 4.0, và băng
thông mạng giữa các node.

In [ ]:
PCIE4 = sach(32, nguon="32 GB/s · 2,04 TB/s · 63,7×")
HBM = sach(2039, nguon="2039 GB/s · 0,002 ms")
theo_sach("bộ nhớ GPU so với PCIe 4.0 (lần)", HBM / PCIE4, sach=63.7, nguon="32 GB/s · 2,04 TB/s · 63,7×")
print(f"nếu dùng 2.04 TB/s đã làm tròn: {sach(2.04, nguon='32 GB/s · 2,04 TB/s · 63,7×') * 1e3 / PCIE4:.2f} lần")
D_1GB = sach(1, nguon="1 GB → 31,2 ms") * 1e9
print(f"1 GB qua PCIe 4.0: {D_1GB / (PCIE4 * 1e9) * 1e3:.2f} ms; sách ghi {sach(31.2, nguon='1 GB → 31,2 ms')} ms")
mang_thap = sach(50, nguon="900 GB/s · 50–400 Gbps (6,25–50 GB/s)")      # Gbps
mang_cao = sach(400, nguon="900 GB/s · 50–400 Gbps (6,25–50 GB/s)")
theo_sach("mạng 50 Gbps (GB/s)", mang_thap / 8, sach=6.25, nguon="900 GB/s · 50–400 Gbps (6,25–50 GB/s)")
theo_sach("mạng 400 Gbps (GB/s)", mang_cao / 8, sach=50, nguon="900 GB/s · 50–400 Gbps (6,25–50 GB/s)")
NVLINK_NODE = sach(900, nguon="900 GB/s · 50–400 Gbps (6,25–50 GB/s)")
print(f"NVLink trong một node nhanh hơn mạng giữa các node từ {NVLINK_NODE / (mang_cao / 8):.0f} "
      f"tới {NVLINK_NODE / (mang_thap / 8):.0f} lần")

63.7 lần là 2039 chia 32. Viết 2039 GB/s thành 2.04 TB/s thì tỉ số thành 63.75, nên làm tròn ra 63.7 hay
63.8 là tuỳ cách viết băng thông. 1 GB chia 32 GB/s là đúng 31.25 ms, và chương làm tròn thành 31.2: một
con số kết thúc bằng 5 bị làm tròn xuống, nên sổ tay in cả hai thay vì kiểm.

### Đo độ trễ và băng thông của một đường copy

CPU của bạn không có PCIe tới GPU, nhưng chép giữa hai vùng nhớ trên máy chủ cũng theo đúng mô hình
$T = L_{lat} + D/BW$. Ô dưới chép những khối từ 256 byte tới 64 MB, rồi khớp hai tham số. Khớp theo sai
số tương đối, để khối nhỏ và khối lớn có trọng số như nhau.

In [ ]:
co_khoi = [2**k for k in range(8, 27, 2)]
t_do = []
for so_byte in co_khoi:
    nguon_ = torch.randn(so_byte // 4)
    dich_ = torch.empty_like(nguon_)
    lap = max(3, min(2000, int(2e7 / so_byte)))        # khối nhỏ: lặp nhiều lần rồi chia
    t_do.append(thoi_gian(lambda: [dich_.copy_(nguon_) for _ in range(lap)], 3) / lap)
D_ = np.array(co_khoi, dtype=float)
T_ = np.array(t_do)
(L_may, nghich_BW), *_ = np.linalg.lstsq(np.stack([np.ones_like(D_), D_], 1) / T_[:, None], np.ones_like(T_), rcond=None)
BW_may = 1 / nghich_BW
do_tren_may("độ trễ cố định mỗi lần chép, L_lat", L_may * 1e6, "µs")
do_tren_may("băng thông chép, BW", BW_may / 1e9, "GB/s")
do_tren_may("cỡ khối mà hai số hạng bằng nhau, L_lat · BW", L_may * BW_may / 1e3, "KB")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.loglog(D_, T_ * 1e6, "o", color=MAU[0], ms=7, label="đo được")
ax.loglog(D_, (L_may + D_ / BW_may) * 1e6, "-", color=MAU[1], lw=1.5, label="khớp: L_lat + D / BW")
ax.loglog(D_, D_ / BW_may * 1e6, ":", color="0.5", lw=1.2, label="chỉ số hạng băng thông")
ax.set_xlabel("số byte mỗi lần chép")
ax.set_ylabel("thời gian (µs)")
ax.set_title("Chép giữa hai vùng nhớ trên máy chủ (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Ở khối nhỏ, đường đo nằm phẳng trên độ trễ cố định: chép nhiều khối nhỏ thì phần lớn thời gian là phụ
trội, đúng như thuế điều phối ở sổ tay 01. Ở khối lớn, nó bám đường băng thông. Hai vùng gặp nhau ở cỡ
khối $L_{lat} \cdot BW$ in ở trên. Bảng 7 chỉ giữ số hạng băng thông, nên nó là chặn dưới, như chương
nói, chứ không phải độ trễ đo được.

## 5. Quy trình dữ liệu phải theo kịp

Chương tính nhịp dữ liệu mà một GPU xử lý 1,000 ảnh mỗi giây cần, thời gian chép một batch qua PCIe 4.0,
và độ sâu hàng đợi khi nạp trước.

In [ ]:
anh_moi_giay = sach(1000, nguon="1.000 ảnh/giây · 224×224 · 150,5 MB/s")
canh = sach(224, nguon="1.000 ảnh/giây · 224×224 · 150,5 MB/s")
kenh = sach(3, nguon="batch 64 FP32 224×224×3 = 38,5 MB → 1,2 ms")
theo_sach("nhịp ảnh thô uint8 (MB/s)", anh_moi_giay * canh * canh * kenh * 1 / 1e6, sach=150.5,
          nguon="1.000 ảnh/giây · 224×224 · 150,5 MB/s")
batch_byte = batch_anh.numel() * FP32_B
theo_sach("batch 64 ảnh FP32 qua PCIe 4.0 (ms)", batch_byte / (PCIE4 * 1e9) * 1e3, sach=1.2,
          nguon="batch 64 FP32 224×224×3 = 38,5 MB → 1,2 ms")
so_cong_nhan = sach(4, nguon="4 worker · prefetch 2 → 8 batch")
nap_truoc = sach(2, nguon="4 worker · prefetch 2 → 8 batch")
theo_sach("batch đã chuẩn bị sẵn tối đa", so_cong_nhan * nap_truoc, sach=8, nguon="4 worker · prefetch 2 → 8 batch")
print(f"bộ nhớ máy chủ cho 8 batch đó: {so_cong_nhan * nap_truoc * batch_byte / 1e6:.0f} MB")

Dòng cuối là cái giá của nạp trước mà chương nhắc: bộ nhớ máy chủ tỉ lệ với lượng dữ liệu xếp hàng.

### Đo: bộ nạp có theo kịp mô hình không

Ô dưới đo hai nhịp trên máy này: bộ nạp dữ liệu (data loader) đọc MNIST ở một tiến trình (`num_workers=0`),
và một bước huấn luyện của một MLP nhỏ. Không có tiến trình công nhân thì nạp và tính chạy nối tiếp, nên
phần thời gian dành cho tính toán là $T_{tính} / (T_{nạp} + T_{tính})$: đó là mức sử dụng của phần tính
toán trong vòng lặp này.

In [ ]:
tap = torchvision.datasets.MNIST(DATA, train=True, download=True, transform=transforms.ToTensor())
tap_con = torch.utils.data.Subset(tap, range(4096))
bo_nap = torch.utils.data.DataLoader(tap_con, batch_size=64, shuffle=True, num_workers=0)
t0 = time.perf_counter()
so_mau = sum(xb.shape[0] for xb, _ in bo_nap)
nhip_nap = so_mau / (time.perf_counter() - t0)

dat_hat_giong()
mlp = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))
toi_uu = torch.optim.SGD(mlp.parameters(), lr=0.01)
xb, yb = next(iter(bo_nap))


def buoc():
    toi_uu.zero_grad()
    F.cross_entropy(mlp(xb), yb).backward()
    toi_uu.step()


nhip_tinh = xb.shape[0] / thoi_gian(buoc, 20)
do_tren_may("nhịp bộ nạp (num_workers=0)", nhip_nap, "ảnh/giây")
do_tren_may("nhịp mô hình tiêu thụ", nhip_tinh, "ảnh/giây")
do_tren_may("phần thời gian dành cho tính toán", 100 * (1 / nhip_tinh) / (1 / nhip_nap + 1 / nhip_tinh), "%")

Nếu nhịp bộ nạp thấp hơn nhịp tiêu thụ, mô hình phải chờ dữ liệu và mức sử dụng tụt xuống dưới một. Đó là
lúc chương nói tới các tiến trình công nhân (worker process) và nạp trước: chúng cho việc nạp chồng lên
việc tính. Nếu bộ nạp đã nhanh hơn, thêm công nhân chỉ thêm phụ trội.

### Gom mẫu thành batch

Cách gom mặc định xếp các mẫu cùng hình dạng thành một trục batch mới. Với chuỗi độ dài thay đổi, phải
đệm mọi chuỗi trong một batch tới chuỗi dài nhất, và phần đệm là byte di chuyển rồi tính mà không mang
thông tin. Ô dưới đếm phần đệm của 1,024 chuỗi ví dụ khi gom theo thứ tự ngẫu nhiên, và khi sắp theo độ
dài trước rồi mới gom.

In [ ]:
dat_hat_giong()
do_dai = torch.randint(8, 257, (1024,))                 # độ dài chuỗi ví dụ
B_GOM = 64


def ti_le_dem(thu_tu):
    nhom = do_dai[thu_tu].view(-1, B_GOM)
    return 1 - nhom.sum().item() / (nhom.max(1).values * B_GOM).sum().item()


ngau_nhien = ti_le_dem(torch.randperm(1024))
sap_xep = ti_le_dem(torch.argsort(do_dai))
print(f"phần đệm khi gom ngẫu nhiên: {100 * ngau_nhien:.1f} %, khi sắp theo độ dài: {100 * sap_xep:.1f} %")
assert sap_xep < ngau_nhien

Sắp theo độ dài làm mất tính ngẫu nhiên của thứ tự, nên trong thực tế người ta thường chỉ sắp trong từng
nhóm lớn rồi trộn các batch. Dù sao, phép đếm cho thấy vì sao chương nói cách gom mẫu ảnh hưởng tới cả bộ
nhớ lẫn thông lượng.

## 6. Ba nguyên lý của `nn.Module`

Chương nêu ba nguyên lý: phát hiện tham số (parameter discovery) tự động, hành vi theo chế độ, và kết
hợp phân cấp cùng tuần tự hóa (serialization). Ô dưới thử từng cái bằng một khẳng định kiểm được.

In [ ]:
class DanhSachThuong(nn.Module):
    def __init__(self):
        super().__init__()
        self.tang = [nn.Linear(4, 4), nn.Linear(4, 2)]     # danh sách Python thường: không được đăng ký


class DanhSachModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.tang = nn.ModuleList([nn.Linear(4, 4), nn.Linear(4, 2)])


print("tham số tìm thấy: danh sách thường", len(list(DanhSachThuong().parameters())),
      "· ModuleList", len(list(DanhSachModule().parameters())))
try:
    torch.optim.SGD(DanhSachThuong().parameters(), lr=0.1)
except ValueError as loi:
    print("bộ tối ưu với danh sách thường:", loi)

bn = nn.BatchNorm1d(4)
print("BatchNorm1d: tham số", [n for n, _ in bn.named_parameters()], "· bộ đệm", [n for n, _ in bn.named_buffers()])
print("từ điển trạng thái chứa cả hai:", list(bn.state_dict()))
assert set(bn.state_dict()) == {n for n, _ in bn.named_parameters()} | {n for n, _ in bn.named_buffers()}

dong_bang = nn.Sequential(nn.Linear(4, 4), nn.ReLU(), nn.Linear(4, 2))
for p in dong_bang[0].parameters():
    p.requires_grad_(False)                             # đóng băng tầng đầu
dong_bang(torch.randn(3, 4)).sum().backward()
print("tầng đóng băng vẫn nằm trong parameters():", len(list(dong_bang.parameters())) == 4,
      "· gradient của nó:", dong_bang[0].weight.grad, "· của tầng sau:", tuple(dong_bang[2].weight.grad.shape))

Một danh sách Python thường giấu các tầng khỏi phép duyệt cây module, nên bộ tối ưu không thấy tham số
nào. Bộ đệm (buffer) như thống kê chạy của chuẩn hóa theo batch (batch normalization) đi cùng module vào
từ điển trạng thái (state dict), nhưng không phải tham số nên không nhận gradient. Đóng băng chỉ tắt
gradient: tham số vẫn được phát hiện.

In [ ]:
dat_hat_giong()
mo_hinh = nn.Sequential(nn.Linear(8, 16), nn.BatchNorm1d(16), nn.ReLU(), nn.Dropout(0.5), nn.Linear(16, 2))
x8 = torch.randn(32, 8)
mo_hinh.train()
khac_nhau = not torch.equal(mo_hinh(x8), mo_hinh(x8))
mo_hinh.eval()                                          # một lời gọi ở gốc
assert all(not m.training for m in mo_hinh.modules())   # cờ đã lan tới mọi module con
print("chế độ huấn luyện: hai lần gọi cho hai kết quả khác nhau:", khac_nhau)
print("chế độ suy luận: hai lần gọi giống hệt nhau:", torch.equal(mo_hinh(x8), mo_hinh(x8)))
assert torch.equal(mo_hinh[3](x8), x8)                  # dropout ở chế độ suy luận là ánh xạ đồng nhất

ghi = {}
moc = mo_hinh[2].register_forward_hook(lambda m, vao, ra: ghi.update(ty_le_khong=(ra == 0).float().mean().item()))
mo_hinh(x8)
moc.remove()
print(f"móc lượt xuôi trên ReLU: {100 * ghi['ty_le_khong']:.0f} % giá trị kích hoạt bằng 0, không sửa dòng mã nào của mô hình")

khoa = list(mo_hinh.state_dict())
print("khoá của từ điển trạng thái:", khoa)
bo_nho_dem = io.BytesIO()
torch.save(mo_hinh.state_dict(), bo_nho_dem)
tai_trong = sum(t.numel() * t.element_size() for t in mo_hinh.state_dict().values())
print(f"tệp lưu {bo_nho_dem.tell():,} byte cho {tai_trong:,} byte tham số và bộ đệm")

khac = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 2))
ket_qua = khac.load_state_dict(mo_hinh.state_dict(), strict=False)
print("nạp vào một mô hình khác cấu trúc: thiếu", ket_qua.missing_keys, "· thừa", ket_qua.unexpected_keys[:3], "…")
P7 = sach(7e9, nguon="7 tỷ · 70 GB = 14 GB + 56 GB")
theo_sach("checkpoint 7 tỷ tham số FP16, phần trọng số (GB)", P7 * 2 / 1e9, sach=14, nguon="7 tỷ · 70 GB = 14 GB + 56 GB")

Khoá của từ điển trạng thái mã hoá vị trí trong cây module (`1.running_mean` là bộ đệm của module con thứ
hai). `load_state_dict` chỉ chép giá trị vào một module đã dựng sẵn và khớp theo khoá: nó không dựng lại
cấu trúc. Hai mô hình ở đây có cùng tầng tuyến tính cuối, nhưng mô hình sau bỏ BatchNorm và Dropout nên
tầng ấy mang khoá `2.weight` thay vì `4.weight`: nó bị báo thiếu, còn `4.weight` bị báo thừa. Một tệp
trọng số không mang theo đồ thị.

## 7. Đặt mô hình lên một đích triển khai

Chương lấy ví dụ một mô hình phát hiện từ khoá (keyword spotting) cho chuông cửa thông minh, chạy trên một
ARM Cortex-M4 có 256 KB RAM và 1 MB Flash, qua một môi trường chạy siêu nhỏ (micro-runtime): một vùng nhớ
cố định, và chỉ những kernel mà mô hình dùng. Ô dưới dựng một **mô hình ví dụ** cùng kiểu (tích chập tách
theo chiều sâu trên một phổ MFCC 49 × 10; đây không phải mô hình của chương), bắt đồ thị của nó bằng
`torch.fx` như ở sổ tay 02, rồi kiểm ba thứ: tập toán tử, Flash cho trọng số, RAM cho giá trị kích hoạt.

In [ ]:
RAM = sach(256, trich="ARM Cortex-M4 with 256 KB RAM and 1 MB Flash") * 1024        # KB nhị phân
FLASH = sach(1, trich="ARM Cortex-M4 with 256 KB RAM and 1 MB Flash") * 1024 * 1024


class TuKhoaNho(nn.Module):
    """Mô hình ví dụ: một tích chập vào, bốn khối tách theo chiều sâu, gộp trung bình, một tầng ra 12 lớp."""

    def __init__(self, c=64, kich_hoat=nn.ReLU):
        super().__init__()
        self.vao = nn.Sequential(nn.Conv2d(1, c, (10, 4), stride=2, padding=(5, 1)), nn.BatchNorm2d(c), kich_hoat())
        lop = []
        for _ in range(4):
            lop += [nn.Conv2d(c, c, 3, padding=1, groups=c), nn.BatchNorm2d(c), kich_hoat(),
                    nn.Conv2d(c, c, 1), nn.BatchNorm2d(c), kich_hoat()]
        self.khoi = nn.Sequential(*lop)
        self.gop = nn.AdaptiveAvgPool2d(1)
        self.ra = nn.Linear(c, 12)

    def forward(self, x):
        return self.ra(torch.flatten(self.gop(self.khoi(self.vao(x))), 1))


def toan_tu(gm):
    """Tên toán tử theo kiểu một môi trường chạy siêu nhỏ, cho mỗi đỉnh tính toán của đồ thị."""
    ten = []
    for dinh in gm.graph.nodes:
        if dinh.op == "call_module":
            m = gm.get_submodule(dinh.target)
            if isinstance(m, nn.Conv2d):
                ten.append("DEPTHWISE_CONV_2D" if m.groups == m.in_channels > 1 else "CONV_2D")
            else:
                ten.append({nn.BatchNorm2d: "BATCH_NORM", nn.ReLU: "RELU", nn.GELU: "GELU",
                            nn.AdaptiveAvgPool2d: "AVERAGE_POOL_2D", nn.Linear: "FULLY_CONNECTED"}[type(m)])
        elif dinh.op == "call_function":
            ten.append("RESHAPE")                       # flatten
    return ten


# giả định: tập kernel mà môi trường chạy siêu nhỏ này có sẵn
CO_SAN = {"CONV_2D", "DEPTHWISE_CONV_2D", "RELU", "AVERAGE_POOL_2D", "RESHAPE", "FULLY_CONNECTED", "SOFTMAX"}
dat_hat_giong()
kws = TuKhoaNho().eval()
gm = fx.symbolic_trace(kws)
mfcc = torch.randn(1, 1, 49, 10)
ShapeProp(gm).propagate(mfcc)
can = Counter(toan_tu(gm))
print("toán tử mô hình cần:", dict(can))
print("không có trong môi trường chạy:", sorted(set(can) - CO_SAN))
gm_gelu = fx.symbolic_trace(TuKhoaNho(kich_hoat=nn.GELU).eval())
print("đổi ReLU thành GELU, thiếu thêm:", sorted(set(toan_tu(gm_gelu)) - CO_SAN))

Mô hình cần bảy loại toán tử, và chỉ những kernel đó phải được liên kết vào môi trường chạy, như chương
nói. Chuẩn hóa theo batch là loại duy nhất không có trong tập giả định. Ở chế độ suy luận nó chỉ là một phép nhân và một
phép cộng với hằng số trên mỗi kênh, nên có thể gộp hẳn vào trọng số của tích chập ngay trước nó, một dạng
tính sẵn hằng số (constant folding):
$W' = W \cdot \gamma / \sqrt{\sigma^2 + \epsilon}$ và $b' = (b - \mu)\,\gamma / \sqrt{\sigma^2 + \epsilon} + \beta$.
Còn GELU thì không gộp được: đó là chỗ độ phủ toán tử (operator coverage) buộc phải đổi mô hình, hoặc đổi
môi trường chạy.

In [ ]:
def gop_bn(tich_chap_, bn_):
    """Một Conv2d mới có kết quả bằng tích_chập rồi BatchNorm (chế độ suy luận)."""
    he = bn_.weight / torch.sqrt(bn_.running_var + bn_.eps)
    moi_ = nn.Conv2d(tich_chap_.in_channels, tich_chap_.out_channels, tich_chap_.kernel_size,
                     tich_chap_.stride, tich_chap_.padding, groups=tich_chap_.groups)
    with torch.no_grad():
        moi_.weight.copy_(tich_chap_.weight * he.view(-1, 1, 1, 1))
        moi_.bias.copy_((tich_chap_.bias - bn_.running_mean) * he + bn_.bias)
    return moi_


with torch.no_grad():                                   # thống kê chạy khác mặc định, để phép gộp có việc
    for m in kws.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.running_mean.uniform_(-0.5, 0.5)
            m.running_var.uniform_(0.5, 2.0)
            m.weight.uniform_(0.5, 1.5)
            m.bias.uniform_(-0.2, 0.2)


def gop_ca_mo_hinh(day):
    lop, i = [], 0
    while i < len(day):
        if isinstance(day[i], nn.Conv2d) and i + 1 < len(day) and isinstance(day[i + 1], nn.BatchNorm2d):
            lop.append(gop_bn(day[i], day[i + 1]))
            i += 2
        else:
            lop.append(day[i])
            i += 1
    return nn.Sequential(*lop)


kws_gop = TuKhoaNho().eval()
kws_gop.vao, kws_gop.khoi, kws_gop.gop, kws_gop.ra = gop_ca_mo_hinh(kws.vao), gop_ca_mo_hinh(kws.khoi), kws.gop, kws.ra
lech = (kws_gop(mfcc) - kws(mfcc)).abs().max().item()
print(f"sau khi gộp: {len(list(kws.modules()))} → {len(list(kws_gop.modules()))} module, lệch đầu ra lớn nhất {lech:.1e}")
assert lech < 1e-4
gm_gop = fx.symbolic_trace(kws_gop)
ShapeProp(gm_gop).propagate(mfcc)
print("không có trong môi trường chạy, sau khi gộp:", sorted(set(toan_tu(gm_gop)) - CO_SAN))
assert not set(toan_tu(gm_gop)) - CO_SAN

Giờ là hai ngân sách. Trọng số nằm ở Flash. Giá trị kích hoạt nằm trong vùng nhớ cố định ở RAM: với một
chuỗi tầng nối tiếp, mỗi tầng cần đầu vào và đầu ra của nó cùng lúc, nên vùng nhớ phải chứa được cặp lớn
nhất. Đây là ước lượng thô, bỏ qua bộ đệm làm việc của từng kernel.

In [ ]:
so_tham_so = sum(p.numel() for p in kws_gop.parameters())
kich_thuoc = [int(np.prod(d.meta["tensor_meta"].shape)) for d in gm_gop.graph.nodes
              if d.op != "output" and "tensor_meta" in d.meta]
cap_lon_nhat = max(a + b for a, b in zip(kich_thuoc, kich_thuoc[1:]))
hang = []
for ten, so_byte in (("FP32", FP32_B), ("INT8", INT8_B)):
    hang.append((ten, so_tham_so * so_byte / 1024, 100 * so_tham_so * so_byte / FLASH,
                 cap_lon_nhat * so_byte / 1024, 100 * cap_lon_nhat * so_byte / RAM))
print(f"mô hình ví dụ: {so_tham_so:,} tham số, cặp kích hoạt lớn nhất {cap_lon_nhat:,} giá trị")
print(pd.DataFrame(hang, columns=["kiểu", "trọng số (KB)", "% Flash 1 MB", "vùng kích hoạt (KB)", "% RAM 256 KB"])
      .to_string(index=False, float_format="{:.1f}".format))

thu_vien = glob.glob(os.path.join(os.path.dirname(torch.__file__), "lib", "*.so*"))
co_thu_vien = sum(os.path.getsize(p) for p in thu_vien)
do_tren_may("thư viện dùng chung của bản PyTorch đang chạy", co_thu_vien / 1e6, "MB")
do_tren_may("so với Flash 1 MB", co_thu_vien / FLASH, "lần")

Mô hình ví dụ vừa cả hai ngân sách. Riêng các thư viện của PyTorch trên máy này đã lớn hơn Flash của
thiết bị nhiều bậc độ lớn, như dòng ⏱ ở trên cho thấy, chưa kể trình thông dịch Python. Đó là ý của chương khi nói một môi trường chạy với
vùng nhớ cố định và mã C/C++ gọn không phải là sở thích, mà là điều kiện để vừa thiết bị. Cạm bẫy chọn
framework theo độ phổ biến nằm đúng ở đây: các thiết bị TinyML nhỏ thường có dưới 256 KB RAM, nên đích triển
khai phải được kiểm trước khi chọn framework để huấn luyện.

In [ ]:
ram_tinyml = sach(256, nguon="256 KB") * 1024
print(f"vùng kích hoạt INT8 của mô hình ví dụ: {cap_lon_nhat * INT8_B / ram_tinyml:.1%} của {ram_tinyml // 1024} KB")
for c in (64, 128, 256, 512):
    m_ = TuKhoaNho(c=c).eval()
    g_ = fx.symbolic_trace(m_)
    ShapeProp(g_).propagate(mfcc)
    kt_ = [int(np.prod(d.meta["tensor_meta"].shape)) for d in g_.graph.nodes if d.op != "output" and "tensor_meta" in d.meta]
    cap_ = max(a + b for a, b in zip(kt_, kt_[1:]))
    print(f"rộng {c:>3} kênh: trọng số INT8 {sum(p.numel() for p in m_.parameters()) / 1024:6.1f} KB, "
          f"vùng kích hoạt INT8 {cap_ / 1024:5.1f} KB, FP32 {cap_ * FP32_B / 1024:6.1f} KB")

Nhân đôi số kênh thì nhân đôi vùng kích hoạt và gần như nhân bốn trọng số, vì tầng tích chập 1×1 có $c^2$
trọng số. Ở 512 kênh, vùng kích hoạt FP32 đã vượt 256 KB RAM, còn INT8 đưa nó về lại trong ngân sách:
kiểu dữ liệu ở phần 3 là đòn bẩy quyết định ở đích này. Nhưng cũng ở 512 kênh, trọng số INT8 đã vượt
Flash 1 MB, mà INT8 đã là kiểu nhỏ nhất ở phần 3: lúc đó phải thu nhỏ chính mô hình.

## Thử thêm

1. Ở ô `layout_bug`, thay `reshape(2, 3, 8, 8)` bằng `permute(0, 3, 2, 1)`, tức đổi chỗ cả hai trục không
   gian. Hình dạng vẫn khớp; đầu ra tích chập có còn đúng không, và vì sao?
2. Ở ô `dtype`, thay hệ số co giãn chung bằng một hệ số cho mỗi nhóm 100 phần tử
   (`x_goc.view(100, 100)`, lấy `max` theo hàng). Sai số INT8 lớn nhất giảm bao nhiêu?
3. Ở ô `kws_model`, xoá `"RELU"` khỏi `CO_SAN` và thêm `"RELU6"`. Mô hình còn chạy được trên môi trường chạy
   này không, và phải đổi gì trong `TuKhoaNho`?

## Tóm lại

* Tensor là một hợp đồng: hình dạng, kiểu, thiết bị và sải bước. Framework chỉ kiểm được hình dạng, nên một
  bố trí sai vẫn chạy và cho kết quả sai.
* Chuyển vị và cắt lát là view miễn phí, nhưng phép đọc một bố trí lệch thì tốn: cùng số byte, khác thời gian.
* Kiểu dữ liệu đổi byte lấy độ chính xác và dải số: INT8 giảm bộ nhớ 4 lần, còn FP16 cần co giãn mất mát
  vì dải số hẹp, điều BF16 không cần.
* Mỗi lần truyền là độ trễ cộng byte chia băng thông. Bảng 7 cho thấy cùng 4 MB mà thời gian lệch nhau hơn
  hai bậc độ lớn giữa PCIe và bộ nhớ trên GPU.
* Ở đích triển khai nhỏ, tập toán tử, Flash và RAM là ràng buộc cứng, và phải được kiểm trước khi chọn
  framework.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Framework — tầng giữa toán học và phần cứng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch07-frameworks/), dựng từ chương
[*ML Frameworks*](https://mlsysbook.ai/vol1/frameworks/frameworks.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_07_ml_frameworks.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.